# llm-traffic-replay: diagnostic endpoint-contract canary
Self-contained runnable payload (v0.6.0, 106 payload files, 1384 pytest cases), unpacked to a fresh driver directory and run against one explicitly selected **pay-per-token** endpoint. Its measured replay schedule is fixed at 0.1 QPS for 12 seconds. With the deterministic bundled seed, this produces one calibration request and one measured replay request after the two representative preflight requests. The illustrative profile declares input p50/p95 of 1,000/2,000 tokens and output-budget p50/p95 of 320/480 tokens; the derived output safety cap is 720 tokens. Compatibility fallbacks/retries make the conservative plan at most 12 physical POST attempts, each requiring separate runtime admission. With the default direct-endpoint `extra_body_json` of `{"reasoning_effort":"none"}`, the bundled offline plan reserves a worst-case 89,202 input tokens/minute and 4,464 output tokens/minute (44.601% and 22.32% of the dated configured limits); changing that control is replanned and can change those bounds. Unrelated workspace traffic remains unknown. The serving API cannot verify the configured Enterprise workspace tier, so confirm that tier and current workspace-wide quota telemetry before setting the paid confirmation. The canary uses the illustrative GLM 5.2 profile and the dated Databricks quota snapshot. The cluster must provide Python 3.10+, NumPy 1.24+, and pytest 7+.

The default `reasoning_effort=none` is the serving-engineering-confirmed top-level request-body control for managed GLM reasoning-off behavior. The same field is confirmed for AI Gateway, but this canary intentionally targets the direct `/serving-endpoints/.../invocations` path. Although the tool can POST a Gateway protocol, its current quota, control-plane, and route-stability binding cannot support an AI Gateway production-capacity claim. Set `extra_body_json` only to request controls documented for the exact model/provider contract. These values are persisted as evidence; credential-like values are rejected.

**What this canary checks:** endpoint binding and stability, per-POST runtime quota admission, streaming, TTFT-on-first-visible-content capture, clean answer completion, usage and cached-token fields, response-model identity, and durable artifact integrity. The separate local mock validation checks the timing instrument against known delays.

**What this canary must never be quoted for:** latency, SLA attainment, throughput, endpoint capacity, or customer workload demand. One short run on an illustrative shape cannot support any of those conclusions. A production benchmark requires a measured workload profile, written targets, an authorized load window, and the intended deployment mode.

In [ ]:
# Cell 1: unpack the embedded runnable payload to a fresh driver directory
import base64, hashlib, json, os, sys, tempfile
from pathlib import Path, PurePosixPath

PACKED_VERSION = "0.6.0"
EXPECTED_PAYLOAD_FILES = 106
PAYLOAD_SHA256 = "5a297099fd47e5768aeefd0965c7085bc706a2fb265dc2b2aea53ad976b7c9c6"
PAYLOAD = "..."

raw_payload = base64.b64decode(PAYLOAD, validate=True)
actual_digest = hashlib.sha256(raw_payload).hexdigest()
if actual_digest != PAYLOAD_SHA256:
    raise RuntimeError("embedded payload checksum mismatch; stop")
# Bootstrap exception: this digest-authenticated payload contains the strict parser itself.
payload_files = json.loads(raw_payload)
if not isinstance(payload_files, dict) or len(payload_files) != EXPECTED_PAYLOAD_FILES:
    raise RuntimeError("embedded payload file count mismatch; stop")
root = Path(tempfile.mkdtemp(prefix="llm-traffic-replay-"))
for rel, text in payload_files.items():
    pure = PurePosixPath(rel)
    if pure.is_absolute() or not pure.parts or ".." in pure.parts or not isinstance(text, str):
        raise RuntimeError(f"unsafe embedded payload entry: {rel!r}")
    p = root.joinpath(*pure.parts)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(text.encode("utf-8"))
preloaded = sorted(name for name in sys.modules if name == "traffic_replay" or name.startswith("traffic_replay."))
if preloaded:
    raise RuntimeError(f"traffic_replay was already imported ({preloaded[:3]}); restart Python and rerun from Cell 1")
os.chdir(root)
sys.path.insert(0, str(root))
import traffic_replay
if (traffic_replay.__version__ != PACKED_VERSION
        or Path(traffic_replay.__file__).resolve().parent != (root / "traffic_replay").resolve()):
    raise RuntimeError("embedded package version or import origin mismatch; stop")
from traffic_replay.artifacts import snapshot_source_state
PACKED_SOURCE_STATE = snapshot_source_state(root / "traffic_replay")
if (PACKED_SOURCE_STATE.get("source_identity_origin") != "embedded_build"
        or PACKED_SOURCE_STATE.get("git_dirty") is not False
        or not isinstance(PACKED_SOURCE_STATE.get("git_commit"), str)
        or not isinstance(PACKED_SOURCE_STATE.get("build_id"), str)):
    raise RuntimeError(f"embedded source provenance is not reconstructible: {PACKED_SOURCE_STATE}")
print("unpacked exact payload to", root, "|", len(payload_files), "files | sha256", actual_digest)
print("payload source commit", PACKED_SOURCE_STATE["git_commit"], "| build id", PACKED_SOURCE_STATE["build_id"])

In [ ]:
# Cell 2: run the full pytest suite (1384 cases) + instrument validation
import json, os, re, subprocess, sys, tempfile
import xml.etree.ElementTree as ET
from importlib.metadata import PackageNotFoundError, version as distribution_version
from pathlib import Path
EXPECTED_PYTEST_CASES = 1384
try:
    pytest_major = int(distribution_version("pytest").split(".", 1)[0])
except (PackageNotFoundError, ValueError, TypeError) as exc:
    raise RuntimeError("cannot determine the installed pytest version; stop") from exc
if pytest_major < 7:
    raise RuntimeError("pytest 7 or newer is required; stop")
try:
    numpy_match = re.match(r"^(\d+)\.(\d+)", distribution_version("numpy"))
except PackageNotFoundError as exc:
    raise RuntimeError("NumPy 1.24 or newer is required; stop") from exc
if not numpy_match or tuple(map(int, numpy_match.groups())) < (1, 24):
    raise RuntimeError("NumPy 1.24 or newer is required; stop")
pytest_env = os.environ.copy()
pytest_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
pytest_env.pop("PYTEST_ADDOPTS", None)
pytest_env.pop("PYTEST_PLUGINS", None)
pytest_env.pop("PYTHONPATH", None)

def run_checked(command, timeout_s, label, env=None):
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=timeout_s, env=env)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f"{label} exceeded {timeout_s} seconds; stop") from exc
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-2000:], file=sys.stderr)
        raise RuntimeError(f"{label} failed with exit code {result.returncode}; stop")
    return result

collect = run_checked([sys.executable, "-m", "pytest", "--collect-only", "-q", "-o", "addopts=", "-p", "no:cacheprovider"], 180, "pytest collection", pytest_env)
nodeids = [line for line in collect.stdout.splitlines() if line.startswith("tests/") and "::" in line]
collection_summary = re.search(r"(?m)^(\d+) tests? collected\b", collect.stdout)
if (not collection_summary or int(collection_summary.group(1)) != len(nodeids)
        or len(nodeids) != len(set(nodeids)) or len(nodeids) != EXPECTED_PYTEST_CASES):
    raise RuntimeError(f"pytest collection mismatch: expected {EXPECTED_PYTEST_CASES}, got {len(nodeids)}; stop")
junit_path = Path(tempfile.mkdtemp(prefix="llm-traffic-replay-pytest-")) / "results.xml"
tests = run_checked([sys.executable, "-m", "pytest", "-q", "-o", "addopts=", "-p", "no:cacheprovider", f"--junitxml={junit_path}"], 1200, "pytest suite", pytest_env)
suites = ET.parse(junit_path).getroot().findall("testsuite")
if len(suites) != 1:
    raise RuntimeError(f"pytest JUnit evidence has {len(suites)} suites, expected one; stop")
junit_counts = {name: int(suites[0].attrib.get(name, "0")) for name in ("tests", "failures", "errors", "skipped")}
if (junit_counts["tests"] != EXPECTED_PYTEST_CASES
        or any(junit_counts[name] != 0 for name in ("failures", "errors", "skipped"))):
    raise RuntimeError(f"pytest JUnit counts disagree with collection: {junit_counts}; stop")
print(tests.stdout[-1200:])
validation_root = tempfile.mkdtemp(prefix="llm-traffic-replay-validation-")
validation_run = run_checked([sys.executable, "-m", "traffic_replay", "validate", "--quiet", "--format", "json", "--port", "0", "--duration", "15", "--tolerance-ms", "120", "--workdir", validation_root], 240, "instrument validation", pytest_env)
from traffic_replay.json_input import loads_strict
validation = loads_strict(validation_run.stdout)
if validation.get("passed") is not True or validation.get("joined_requests", 0) < 1:
    raise RuntimeError("instrument validation did not produce passing joined evidence; stop")
print(json.dumps(validation, indent=2, allow_nan=False))

In [ ]:
# Cell 3: require an explicit endpoint and durable Volume destination
import urllib.parse, uuid
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

dbutils.widgets.text("endpoint_name", "", "Pay-per-token endpoint name (required)")
dbutils.widgets.text("artifact_volume_path", "", "Durable /Volumes/... artifact root (required)")
dbutils.widgets.text("extra_body_json", '{"reasoning_effort":"none"}', "Direct managed endpoint request controls (JSON object)")
dbutils.widgets.dropdown("confirm_paid_canary", "NO", ["NO", "RUN"], "Confirm paid canary with 12-second measured schedule")
ENDPOINT = dbutils.widgets.get("endpoint_name").strip()
artifact_value = dbutils.widgets.get("artifact_volume_path").strip()
from traffic_replay.json_input import loads_strict
try:
    EXTRA_BODY = loads_strict(dbutils.widgets.get("extra_body_json"))
except ValueError as exc:
    raise ValueError("extra_body_json must be valid JSON") from exc
from traffic_replay.client import validate_extra_body_safety
validate_extra_body_safety(EXTRA_BODY)
if dbutils.widgets.get("confirm_paid_canary") != "RUN":
    raise ValueError("set confirm_paid_canary to RUN only after confirming Enterprise workspace tier/current workspace-wide quota telemetry and reviewing two preflight requests, one calibration request, and one measured replay request on the fixed 0.1 QPS/12-second schedule; physical retries/fallbacks remain runtime-admitted")
if not ENDPOINT:
    raise ValueError("set the endpoint_name widget explicitly; no endpoint is auto-selected")
if (ENDPOINT in (".", "..") or "/" in ENDPOINT or "\\" in ENDPOINT
        or any(ord(char) < 32 or ord(char) == 127 for char in ENDPOINT)):
    raise ValueError("endpoint_name contains unsafe path characters")
ENDPOINT_PATH_NAME = urllib.parse.quote(ENDPOINT, safe="-_.~")
artifact_path = PurePosixPath(artifact_value)
if (not artifact_value.startswith("/Volumes/") or len(artifact_path.parts) < 5
        or artifact_path.parts[1] != "Volumes" or ".." in artifact_path.parts):
    raise ValueError("artifact_volume_path must be /Volumes/<catalog>/<schema>/<volume>[/subdir]")
ARTIFACT_ROOT = Path(artifact_value)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
CANARY_OUTPUT_BASE = ARTIFACT_ROOT / f"glm52-canary-{stamp}-{uuid.uuid4().hex[:12]}"
if CANARY_OUTPUT_BASE.exists():
    raise RuntimeError(f"fresh artifact base already exists: {CANARY_OUTPUT_BASE}")

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
browser_host = str(ctx.browserHostName().get()).strip()
TOKEN = str(ctx.apiToken().get()).strip()
if not browser_host or "/" in browser_host or not TOKEN:
    raise RuntimeError("workspace host or ambient notebook token is unavailable")
HOST = "https://" + browser_host
INVOCATION_PATH = f"/serving-endpoints/{ENDPOINT_PATH_NAME}/invocations"
from traffic_replay.client import validate_bearer_transport
from traffic_replay.config_validation import validate_rate_limits
from traffic_replay.endpoint_meta import fetch_endpoint_metadata, rate_limit_endpoint_binding
validate_bearer_transport(HOST)
RATE_LIMITS_PATH = Path("configs/rate_limits_databricks_glm_5_2_enterprise_p2t_2026-08-07.json")
RATE_LIMITS = loads_strict(RATE_LIMITS_PATH.read_bytes())
validate_rate_limits(RATE_LIMITS)
selected = fetch_endpoint_metadata(HOST, INVOCATION_PATH, TOKEN, timeout=15.0)
if not isinstance(selected, dict):
    raise RuntimeError("bounded endpoint discovery did not return valid metadata; stop")
binding = rate_limit_endpoint_binding(RATE_LIMITS, selected, INVOCATION_PATH)
if binding.get("binding_complete") is not True:
    raise RuntimeError(f"selected endpoint does not match the dated GLM 5.2 pay-per-token quota scope: {binding.get('reasons')}")
print("GLM 5.2 endpoint/model/deployment shape matches the quota snapshot:", ENDPOINT)
print("workspace tier remains a configured assertion; confirm Enterprise tier and current workspace-wide quota telemetry")
print("provider request-control keys:", sorted(EXTRA_BODY))
print("durable artifact base:", CANARY_OUTPUT_BASE)

In [ ]:
# Cell 4: quota-planned GLM 5.2 endpoint-contract canary
import json, sys
from pathlib import Path
from traffic_replay.json_input import loads_strict

canary_command = [
    sys.executable, "-m", "traffic_replay", "benchmark",
    "--host", HOST, "--endpoint", ENDPOINT,
    "--profile", "configs/profile_glm52_canary_illustrative.json",
    "--fixed-rate", "0.1", "--duration", "12",
    "--max-concurrency", "1", "--max-pending-requests", "1",
    "--ttft-definition", "first_visible",
    "--rate-limits", "configs/rate_limits_databricks_glm_5_2_enterprise_p2t_2026-08-07.json",
    "--out-dir", str(CANARY_OUTPUT_BASE),
    "--title", f"GLM 5.2 endpoint-contract canary: {ENDPOINT}",
    "--label", "DIAGNOSTIC CANARY ONLY: no performance, SLA, throughput, capacity, or workload-demand conclusion.",
    "--fail-on", "none", "--format", "json",
]
if EXTRA_BODY:
    canary_command.extend(["--extra-body", json.dumps(EXTRA_BODY, separators=(",", ":"), allow_nan=False)])
canary_env = pytest_env.copy()  # strips inherited Python path/plugin overrides
canary_env["DATABRICKS_TOKEN"] = TOKEN
paid_canary = run_checked(canary_command, 600, "paid endpoint-contract canary", canary_env)
if paid_canary.stderr.strip():
    print(paid_canary.stderr.rstrip())
summary_from_cli = loads_strict(paid_canary.stdout)
if not isinstance(summary_from_cli, dict):
    raise RuntimeError("benchmark JSON output was not an object; stop")
run_candidates = sorted(path for path in CANARY_OUTPUT_BASE.iterdir()
                        if path.is_dir() and (path / ".traffic-replay-complete").is_file())
if len(run_candidates) != 1:
    raise RuntimeError(f"expected one sealed measured artifact, found {len(run_candidates)}; stop")
run_dir = run_candidates[0]
setup_root = CANARY_OUTPUT_BASE.parent / (CANARY_OUTPUT_BASE.name + "-setup-traffic")
setup_candidates = sorted(path for path in setup_root.iterdir()
                          if path.is_dir() and (path / ".traffic-replay-complete").is_file())
if len(setup_candidates) != 1:
    raise RuntimeError(f"expected one sealed setup-traffic artifact, found {len(setup_candidates)}; stop")
setup_dir = setup_candidates[0]
print("sealed setup and measured artifacts await contract verification:", setup_dir, run_dir)

In [ ]:
# Cell 5: verify diagnostic contract evidence without issuing a performance verdict
import hashlib
from pathlib import Path
from traffic_replay.json_input import loads_strict
from traffic_replay.run_verification import verify_run_output
setup_verification = verify_run_output(setup_dir)
if (setup_verification.get("source_reconstructibility", {}).get("reconstructible") is not True
        or setup_verification.get("decision", {}).get("evidence_integrity", {}).get("code") != "VERIFIED"
        or setup_verification.get("decision", {}).get("endpoint_capacity", {}).get("code") != "NOT_EVALUATED"
        or setup_verification.get("summary", {}).get("setup_traffic", {}).get("outcome") != "preflight_passed"):
    raise RuntimeError("setup traffic is not a verified diagnostic-only preflight artifact; stop")
required = ["start.json", "requests.jsonl", "summary.json", "report.md", "report.html", "manifest.json", ".traffic-replay-complete"]
missing = [name for name in required if not (run_dir / name).is_file()]
if missing:
    raise RuntimeError(f"artifact set is incomplete, missing: {missing}")
manifest_raw = (run_dir / "manifest.json").read_bytes()
manifest = loads_strict(manifest_raw)
complete = loads_strict((run_dir / ".traffic-replay-complete").read_bytes())
manifest_digest = hashlib.sha256(manifest_raw).hexdigest()
artifact_id = manifest.get("artifact_id")
if (manifest.get("manifest_schema_version") != 3 or complete.get("status") != "complete"
        or complete.get("manifest_sha256") != manifest_digest
        or complete.get("manifest_bytes") != len(manifest_raw)
        or not isinstance(artifact_id, str) or not artifact_id
        or complete.get("artifact_id") != artifact_id):
    raise RuntimeError("completion marker does not bind the v3 manifest; stop")
artifacts = manifest.get("artifacts")
if not isinstance(artifacts, dict) or not artifacts:
    raise RuntimeError("manifest has no artifact bindings; stop")
required_bound = {"start.json", "requests.jsonl", "summary.json", "report.md", "report.html"}
if not required_bound.issubset(artifacts):
    raise RuntimeError(f"manifest omits required artifact bindings: {sorted(required_bound - artifacts.keys())}")
for name, expected in artifacts.items():
    if Path(name).name != name or not isinstance(expected, dict):
        raise RuntimeError(f"unsafe manifest artifact entry: {name!r}")
    raw = (run_dir / name).read_bytes()
    if expected.get("bytes") != len(raw) or expected.get("sha256") != hashlib.sha256(raw).hexdigest():
        raise RuntimeError(f"artifact integrity check failed: {name}")
requests_expected = artifacts.get("requests.jsonl", {}).get("row_count")
requests_raw = (run_dir / "requests.jsonl").read_bytes()
if requests_raw and not requests_raw.endswith(b"\n"):
    raise RuntimeError("requests.jsonl has an incomplete final record")
request_lines = requests_raw.splitlines()
if any(not line.strip() for line in request_lines):
    raise RuntimeError("requests.jsonl contains an empty record")
replay_rows = 0
for line_number, line in enumerate(request_lines, 1):
    try:
        request_row = loads_strict(line)
    except ValueError as exc:
        raise RuntimeError(f"requests.jsonl row {line_number} is invalid JSON") from exc
    if not isinstance(request_row, dict):
        raise RuntimeError(f"requests.jsonl row {line_number} is not an object")
    replay_rows += request_row.get("phase") == "replay"
if (not isinstance(requests_expected, int) or isinstance(requests_expected, bool)
        or requests_expected < 0 or requests_expected != len(request_lines)
        or requests_expected != complete.get("request_rows")):
    raise RuntimeError("request row count disagrees across manifest and completion marker")
summary = loads_strict((run_dir / "summary.json").read_bytes())
if not isinstance(summary, dict):
    raise RuntimeError("sealed summary is not an object; stop")
if summary != summary_from_cli:
    raise RuntimeError("CLI JSON and sealed summary.json disagree; stop")
measured_verification = verify_run_output(run_dir)
if (measured_verification.get("source_reconstructibility", {}).get("reconstructible") is not True
        or measured_verification.get("decision", {}).get("evidence_integrity", {}).get("code") != "VERIFIED"
        or measured_verification.get("summary") != summary):
    raise RuntimeError("measured artifact failed integrity, source, or semantic verification; stop")
del measured_verification  # do not retain or publish a capacity classification for this diagnostic canary
start = loads_strict((run_dir / "start.json").read_bytes())
source = start.get("source") if isinstance(start, dict) else None
for source_field in ("git_commit", "git_dirty", "git_status_sha256", "source_tree_sha256", "source_files", "package_version", "build_id", "source_identity_origin", "embedded_provenance_error"):
    if not isinstance(source, dict) or source.get(source_field) != PACKED_SOURCE_STATE.get(source_field):
        raise RuntimeError(f"run source provenance disagrees with unpacked payload at {source_field}; stop")
requests_total = summary.get("requests_total")
answers = summary.get("answers")
visible_ttft = summary.get("ttfv_ms")
usage_coverage = (summary.get("throughput") or {}).get("usage_coverage")
cache = summary.get("achieved_cache_fraction") or {}
if (not isinstance(requests_total, int) or isinstance(requests_total, bool) or requests_total < 1
        or replay_rows != requests_total or summary.get("requests_ok") != requests_total
        or summary.get("requests_failed") != 0):
    raise RuntimeError("diagnostic canary had missing or failed replay requests; stop")
if (not isinstance(answers, dict) or answers.get("judged") != requests_total
        or answers.get("answered") != requests_total or answers.get("answer_rate") != 1.0
        or answers.get("stream_incomplete") != 0 or answers.get("parse_errors") != 0):
    raise RuntimeError("not every replay request produced one clean readable answer; stop")
if (not isinstance(visible_ttft, dict) or visible_ttft.get("n") != requests_total
        or visible_ttft.get("missing") != 0):
    raise RuntimeError("first-visible TTFT was not measured for every replay request; stop")
if usage_coverage != 1.0:
    raise RuntimeError("endpoint token-usage coverage was not 100%; stop")
cache_sources = cache.get("source_fields")
if (cache.get("reported_for_n") != requests_total or cache.get("coverage") != 1.0
        or not isinstance(cache_sources, list) or not cache_sources
        or any(source in ("NOT REPORTED BY ENDPOINT", "SOURCE FIELD NOT RECORDED") for source in cache_sources)):
    raise RuntimeError("cached-token usage and its source field were not reported for every replay request")
run_meta = summary.get("run") or {}
identity = summary.get("response_identity") or {}
admission = summary.get("runtime_quota_admission") or {}
if run_meta.get("ttft_definition") != "first_visible":
    raise RuntimeError("sealed run did not preserve first_visible TTFT semantics; stop")
if run_meta.get("endpoint_metadata_stability") != "stable":
    raise RuntimeError(f"pre-run/post-drain endpoint stability was not established: {run_meta.get('endpoint_metadata_stability')!r}")
if identity.get("status") != "bound" or identity.get("invalid") is not None:
    raise RuntimeError(f"response-model identity was not bound to the selected endpoint: {identity}")
if (admission.get("status") != "enforced" or admission.get("tripped") is not False
        or admission.get("denied_attempts_in_captured_rows") != 0
        or admission.get("invariant_errors") != []):
    raise RuntimeError(f"runtime quota admission evidence was not cleanly enforced: {admission}")
report_markdown = (run_dir / "report.md").read_text(encoding="utf-8")
report_html = (run_dir / "report.html").read_text(encoding="utf-8")
if any(claim in rendered for rendered in (report_markdown, report_html)
       for claim in ("HELD_AT_TESTED_LOAD", "NOT_HELD_AT_TESTED_LOAD", "Tested load held")):
    raise RuntimeError("diagnostic canary report attempted a tested-load capacity verdict; stop")
print(report_markdown)
print("DIAGNOSTIC ENDPOINT-CONTRACT CANARY PASSED")
print("This checks the instrument/endpoint contract only. It is not a latency, SLA, throughput, capacity, or customer-demand result.")
print("verified sealed setup artifact:", setup_dir)
print("measured artifact passed integrity and semantic verification:", run_dir, "| manifest sha256", manifest_digest)
print("The verifier's capacity classification is intentionally not published by this diagnostic notebook.")